

Set `LOGGED_IN = True`

Set `LOGGED_IN = False`

In [49]:
# setup your preferences here
LOGGED_IN = True  # True = use saved login, False = guest browser
TIME_BUDGET_SECONDS = 6  # 0 = legacy mode (find first solution fast), >0 = utilize full time budget (e.g., 8 = exactly 8 seconds)

print(f'Mode: {"Logged in" if LOGGED_IN else "Guest mode"}')


Mode: Logged in


## 1. Checking the environment


In [50]:
import sys
import os
from pathlib import Path

repo_root = Path.cwd()
sys.path.insert(0, str(repo_root))

print('Python:', sys.executable)
print('Version:', sys.version)
print('Repo root:', repo_root)
print()

# checking installed packages
packages = ['playwright', 'torch', 'torchvision', 'cv2', 'numpy', 'PIL']
for pkg in packages:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', 'ok')
        print(f'  {pkg:15s} {ver}')
    except ImportError:
        print(f'  {pkg:15s} MISSING')

Python: c:\Users\shiva\Documents\Zip\linkedin_zip_solver\.venv\Scripts\python.exe
Version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
Repo root: c:\Users\shiva\Documents\Zip\linkedin_zip_solver

  playwright      ok
  torch           2.12.1+cpu
  torchvision     0.27.1+cpu
  cv2             5.0.0
  numpy           2.5.1
  PIL             12.3.0


## 2. Installing dependencies

Let's install any missing packages automatically so you don't have to.

In [51]:
import subprocess

requirements = [
    'playwright>=1.40.0',
    'torch>=2.1.0',
    'torchvision>=0.16.0',
    'opencv-python>=4.8.0',
    'numpy>=1.24.0',
    'pillow>=10.0.0',
]

missing = []
for req in requirements:
    pkg = req.split('>=')[0]
    try:
        __import__(pkg if pkg != 'opencv-python' else 'cv2')
    except ImportError:
        missing.append(req)

if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing)
    print('Done. Please restart the kernel and re-run this cell.')
else:
    print('All dependencies are installed.')

# making sure playwright chromium is installed
try:
    subprocess.check_call([sys.executable, '-m', 'playwright', 'install', 'chromium'])
except Exception as e:
    print('Playwright install note:', e)

Installing missing packages: ['pillow>=10.0.0']
Done. Please restart the kernel and re-run this cell.


## 3. Loading modules

Let's make sure all the code loads up fine.

In [52]:
import main
import model
import solver
import vision
import executor
import train

print('All modules imported successfully:')


All modules imported successfully:


## 4. Testing the CNN

Let's try a quick test inference to make sure the model is working.

In [53]:
from model import load_model, NUM_CHANNELS, HAS_TORCH
import torch

print(f'PyTorch available: {HAS_TORCH}')

if HAS_TORCH:
    model_obj = load_model()
    if model_obj is not None:
        params = sum(p.numel() for p in model_obj.parameters())
        print(f'Model loaded: {params:,} parameters')
        
# try a dummy inference
        dummy = torch.zeros(1, NUM_CHANNELS, 8, 8)
        policy, value = model_obj.predict(dummy)
        print(f'Policy probabilities: {policy.tolist()}')
        print(f'Value score: {value:.4f}')
    else:
        print('Model not loaded (torch available but load failed).')
else:
    print('torch not installed — solver will use plain DFS.')

PyTorch available: True
Model loaded: 95,464 parameters
Policy probabilities: [0.015581503510475159, 0.4423932731151581, 0.023964038118720055, 0.5180611610412598]
Value score: -0.8955


## 5. Quick solver test

Let's run a few random test boards to ensure the solver logic is solid.

In [54]:
import time
import random
import numpy as np
from vision import generate_synthetic_state
from solver import ZipSolver, SolverConfig
from executor import validate_path, build_drag_path, ExecutorConfig

config = SolverConfig(
    max_iterations=2_000_000,
    use_cnn_prior=True,
    use_heuristic_prior=True,
    use_deadlock_check=True,
    verbose=False,
)

all_passed = True
for seed in range(5):
    state, true_path = generate_synthetic_state(rows=6, cols=6, seed=seed)
    solver_inst = ZipSolver(state, model=model_obj, config=config)
    t0 = time.time()
    path = solver_inst.solve()
    elapsed = time.time() - t0
    
    if path is None:
        print(f'Seed {seed}: FAIL — no solution')
        all_passed = False
        continue
    
    ok = (validate_path(path) and
          path[0] == state.start and
          path[-1] == state.goal and
          len(set(path)) == len(path) and
          len(path) == len(true_path))
    
    status = 'PASS' if ok else 'FAIL'
    if not ok:
        all_passed = False
    print(f'Seed {seed}: {status} — {len(path)} cells, '
          f'{solver_inst.nodes_explored} nodes, {elapsed:.3f}s')

print(f'\nSolver test: {"ALL PASSED" if all_passed else "SOME FAILED"}')

Seed 0: PASS — 21 cells, 21 nodes, 0.010s
Seed 1: PASS — 20 cells, 20 nodes, 0.009s
Seed 2: PASS — 23 cells, 23 nodes, 0.006s
Seed 3: PASS — 23 cells, 55 nodes, 0.019s
Seed 4: PASS — 20 cells, 20 nodes, 0.006s

Solver test: ALL PASSED


## 6. Testing DOM parsing

Just making sure we can parse the board layout correctly from the page.

In [55]:
from vision import build_state_from_dom, CellType

# building a dummy 4x4 board with some walls
rows, cols = 4, 4
numbered = {0: 1, 3: 2, 15: 3}  # start, mid, goal
wall_indices = {5, 7, 12}
active_bg = 'rgb(255, 255, 255)'
wall_bg = 'rgb(230, 220, 200)'

cells_data = []
for r in range(rows):
    for c in range(cols):
        idx = r * cols + c
        if idx in wall_indices:
            bg, num = wall_bg, None
        elif idx in numbered:
            bg, num = active_bg, numbered[idx]
        else:
            bg, num = active_bg, None
        cells_data.append({
            'idx': idx, 'number': num,
            'ariaLabel': str(num) if num else None,
            'bg': bg, 'x': c * 50, 'y': r * 50, 'w': 50, 'h': 50,
        })

dom_state = {
    'count': rows * cols,
    'cells': cells_data,
    'gridRect': {'x': 0, 'y': 0, 'w': cols * 50, 'h': rows * 50},
}

state = build_state_from_dom(dom_state)
print(f'Parsed grid ({state.shape[0]}x{state.shape[1]}):')
for r in range(rows):
    row_str = '  '
    for c in range(cols):
        ct = state.grid[r, c]
        if ct == int(CellType.WALL):
            row_str += ' W '
        elif ct == int(CellType.START):
            row_str += ' S '
        elif ct == int(CellType.GOAL):
            row_str += ' G '
        elif ct == int(CellType.NUMBERED):
            for cell in state.cells:
                if cell.row == r and cell.col == c:
                    row_str += f' {cell.number} '
                    break
        else:
            row_str += ' . '
    print(row_str)

print(f'Start: {state.start}, Goal: {state.goal}')
print(f'Waypoints: {state.numbers.tolist()}')
print(f'Walls detected: {sum(1 for r in range(rows) for c in range(cols) if state.grid[r,c] == int(CellType.WALL))}')

Parsed grid (4x4):
   S  .  .  2 
   .  W  .  W 
   .  .  .  . 
   W  .  .  G 
Start: (0, 0), Goal: (3, 3)
Waypoints: [1, 2, 3]
Walls detected: 3


## 7. Training (Optional)

If you don't have the model weights yet, this will train the model for a couple of minutes on some random games.

In [56]:
from pathlib import Path
weights_path = Path('zip_cnn.pth')
if weights_path.exists():
    print(f'Weights already exist: {weights_path} ({weights_path.stat().st_size / 1024:.1f} KB)')
    print('To retrain, delete the file and re-run this cell.')
else:
    print('Training CNN...')
    import subprocess
    result = subprocess.run(
        [sys.executable, 'train.py', '--num-games', '200', '--epochs', '15'],
        cwd=str(repo_root), text=True, capture_output=True
    )
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
    if weights_path.exists():
        print(f'Training complete: {weights_path.stat().st_size / 1024:.1f} KB')

Weights already exist: zip_cnn.pth (386.3 KB)
To retrain, delete the file and re-run this cell.


## 8. Let's solve!

Run this to launch the browser and solve the actual puzzle.

**Tip:** Wait for the puzzle board to fully load in the browser before the solver kicks in.

In [57]:
import subprocess
if LOGGED_IN:
    print('Logged in — Solving your daily LinkedIn Zip puzzle...')
    print('Using your saved browser profile.')
else:
    print('Guest mode — Opening a fresh browser (not logged in)...')
    print('Navigate to the puzzle and click Start Game. Unlimited plays!')

print()

cmd = [sys.executable, 'main.py', '--solve-only', '--skip-setup']
if not LOGGED_IN:
    cmd.append('--guest')
if TIME_BUDGET_SECONDS > 0:
    cmd.append('--time-budget')
    cmd.append(str(TIME_BUDGET_SECONDS))

result = subprocess.run(
    cmd,
    cwd=str(repo_root),
    text=True,
    capture_output=True
)
print(result.stdout)
if result.returncode != 0:
    print('\nExit code:', result.returncode)
    if result.stderr:
        print('\nERROR LOG:\n', result.stderr.strip())

Logged in — Solving your daily LinkedIn Zip puzzle...
Using your saved browser profile.


Starting vision and state capture...
Read 8x8 grid from the page, 16 waypoints
Found a 8x8 puzzle board!
Starting at (3, 7) and aiming for (5, 4).
We need to hit these waypoints: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

Running Hybrid DFS + Heuristic solver...
Figuring out the best path with time budget 6.0s...
  [1 solutions] Best: 64 cells at 1.11s/6.0s (nodes: 8920)
Explored 14167 possibilities in 1.81s, 1 solutions found.
Found the answer! The path is 64 cells long.

Beginning autonomous input execution...
Execution was successful.

I'll leave the browser open for an hour so you can check it out.
Feel free to close the window when you're done.

